# 🧠 Auto QL

**Categoría:** 🧠 Deep Learning `(deep)`  
**Módulo:** `likelihood.models.deep.rl`  
**Clase:** `AutoQL`  
**Fuente:** `likelihood/models/deep/rl.py:127`

> AutoQL: A reinforcement learning agent using Q-learning with Epsilon-greedy policy.


## 📖 Descripción

AutoQL: A reinforcement learning agent using Q-learning with Epsilon-greedy policy.

This class implements a Q-learning agent with:
- Epsilon-greedy policy for exploration
- Replay buffer for experience replay
- Automatic model version handling for TensorFlow


## 🔧 Parámetros

| Parámetro | Tipo | Por defecto | Descripción |
| --- | --- | --- | --- |
| `env` | `Any` | `—` | — |
| `model` | `tf.keras.Model` | `—` | — |
| `maxlen` | `int` | `2000` | — |


## 📋 Métodos disponibles

| Método | Origen | Descripción |
| --- | --- | --- |
| `epsilon_greedy_policy(state: np.ndarray, action: int, epsilon: float = 0.0)` | propio | Epsilon-greedy policy for action selection. |
| `play_one_step(state: np.ndarray, action: int, epsilon: float)` | propio | Perform one step in the environment and add experience to buffer. |
| `train(x_data, y_data, optimizer: str = 'adam', loss_fn: str = 'mse', num_episodes: int = 50, num_steps: int = 100, gamma: float = 0.7, batch_size: int = 32, patience: int = 10, alpha: float = 0.01)` | propio | Train the agent for a fixed number of episodes. |


## 🧩 Constructor

```python
from likelihood.models.deep.rl import AutoQL

AutoQL(env, model: tf.keras.Model, maxlen: int = 2000)
```


## ⚡ Inicio rápido

```python
from likelihood.models.deep.rl import AutoQL

model = AutoQL(env=None, model=None)

# Opcionales: maxlen=2000
```


In [ ]:
from likelihood.models.deep.rl import AutoQL

model = AutoQL(env=None, model=None)

# Opcionales: maxlen=2000

## 🎛️ Ejemplo interactivo

Mueve los controles `maxlen` para reconfigurar el modelo.

> Requiere `ipywidgets` (`pip install ipywidgets`).


In [ ]:
# Ejemplo interactivo de Auto QL
# Generado automáticamente a partir de la firma real de la clase.
from likelihood.models.deep.rl import AutoQL
from ipywidgets import IntSlider, interact

# 1) Datos de entrada requeridos: ajústalos a tu problema
env = None  # Any
model = None  # tf.keras.Model

# 2) Controles interactivos
widgets = {
    "maxlen": IntSlider(value=2000, min=0, max=100, step=1, description="maxlen"),
}


def run_example(**overrides):
    """Construye AutoQL con los valores elegidos."""
    values = {
        "maxlen": overrides["maxlen"],
        "env": env,
        "model": model,
    }
    model = AutoQL(**values)
    print("✅ Modelo construido:", type(model).__name__)
    return model


interact(run_example, **widgets);

## 🧾 Código fuente

<details>
<summary><code>likelihood/models/deep/rl.py</code></summary>

```python
class AutoQL:
    """AutoQL: A reinforcement learning agent using Q-learning with Epsilon-greedy policy.

    This class implements a Q-learning agent with:
    - Epsilon-greedy policy for exploration
    - Replay buffer for experience replay
    - Automatic model version handling for TensorFlow
    """

    def __init__(
        self,
        env: Any,
        model: tf.keras.Model,
        maxlen: int = 2000,
    ) -> None:
        """Initialize AutoQL agent.

        Parameters
        ----------
        env : `Any`
            The environment to interact with
        model : `tf.keras.Model`
            The Q-network model
        """
        self.env = env
        self.model = model
        self.maxlen = maxlen
        self.replay_buffer: deque = deque(maxlen=self.maxlen)

    def epsilon_greedy_policy(
        self, state: np.ndarray, action: int, epsilon: float = 0.0
    ) -> Tuple[Any, int, float, Any, bool]:
        """Epsilon-greedy policy for action selection.

        Parameters
        ----------
        state : `np.ndarray`
            Current state.
        action : `int`
            Expected action to process.
        epsilon : `float`
            Exploration probability. By default it is set to `0.0`

        Returns
        -------
        tuple : (state, action, reward, next_action, done)
        """
        current_state, value, reward, next_action, done = self.env.step(state, action)

        if np.random.rand() > epsilon:
            state = np.asarray(state).astype(np.float32)
            return current_state, value, reward, next_action, done
        step_ = random.sample(self.env.get_transitions(), 1)
        _state, greedy_action, _reward, _next_action, _done = zip(*step_)

        return _state[0], greedy_action[0], _reward[0], _next_action[0], _done[0]

    def play_one_step(
        self, state: np.ndarray, action: int, epsilon: float
    ) -> Tuple[Any, int, float, Any, int]:
        """Perform one step in the environment and add experience to buffer.

        Parameters
        ----------
        state : `np.ndarray`
            Current state
        action : `int`
            Expected action to process.

        epsilon : `float`
            Exploration probability.

        Returns
        -------
        tuple : (state, action, reward, next_action, done)
        """
        current_state, greedy_action, reward, next_action, done = self.epsilon_greedy_policy(
            state, action, epsilon
        )

        done = 1 if done else 0

        # Add experience to replay buffer
        self.replay_buffer.append(
            (
                current_state,  # Previous state
                greedy_action,  # Current action
                reward,  # Reward
                next_action,  # Next action
                done,  # Done flag
            )
        )

        return current_state, greedy_action, reward, next_action, done

    @tf.function
    def _training_step(self) -> float:
        """Perform one training step using experience replay.

        Returns
        -------
        float : Training loss
        """
        batch_ = random.sample(self.replay_buffer, self.batch_size)
        states, actions, rewards, next_actions, dones = zip(*batch_)
        states = np.array(states).reshape(self.batch_size, -1)
        actions = np.array(actions).reshape(
            self.batch_size,
        )
        rewards = np.array(rewards).reshape(
            self.batch_size,
        )
        max_next_Q_values = np.array(next_actions).reshape(self.batch_size, -1)
        dones = np.array(dones).reshape(
            self.batch_size,
        )
        target_Q_values = rewards + (1 - dones) * self.gamma * max_next_Q_values

        actions = tf.convert_to_tensor(actions, dtype=tf.int32)
        states = tf.convert_to_tensor(states, dtype=tf.float32)
        target_Q_values = tf.convert_to_tensor(target_Q_values, dtype=tf.float32)

        with tf.GradientTape() as tape:
            all_Q_values = self.model(states)
            indices = tf.stack([tf.range(tf.shape(actions)[0]), actions], axis=1)
            Q_values = tf.gather_nd(all_Q_values, indices)
            loss = tf.reduce_mean(self.loss_fn(target_Q_values, Q_values))
        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss

    def train(
        self,
        x_data: Any,
        y_data: Any,
        optimizer: str = "adam",
        loss_fn: str = "mse",
        num_episodes: int = 50,
        num_steps: int = 100,
        gamma: float = 0.7,
        batch_size: int = 32,
        patience: int = 10,
        alpha: float = 0.01,
    ) -> None:
        """Train the agent for a fixed number of episodes.

        Parameters
        ----------
        optimizer : `str`
            The optimizer for training (e.g., `sgd`). By default it is set to `adam`.
        loss_fn : `str`
            The loss function. By default it is set to `mse`.
        num_episodes : `int`
            Total number of episodes to train. By default it is set to `50`.
        num_steps : `int`
            Steps per episode. By default it is set to `100`. If `num_steps` is less than `self.env.maxlen`, then the second will be chosen.
        gamma : `float`
            Discount factor. By default it is set to `0.7`.
        batch_size : `int`
            Size of training batches. By default it is set to `32`.
        patience : `int`
            How many episodes to wait for improvement.
        alpha : `float`
            Trade-off factor between loss and reward.
        """
        rewards: List[float] = []
        self.best_weights: Optional[List[np.ndarray]] = None
        self.best_loss: float = float("inf")

        optimizers = {
            "sgd": tf.keras.optimizers.SGD(),
            "adam": tf.keras.optimizers.Adam(),
            "adamw": tf.keras.optimizers.AdamW(),
            "adadelta": tf.keras.optimizers.Adadelta(),
            "rmsprop": tf.keras.optimizers.RMSprop(),
        }
        self.optimizer = optimizers[optimizer]
        losses = {
            "huber": tf.keras.losses.Huber(),
            "mse": tf.keras.losses.MeanSquaredError(),
            "mae": tf.keras.losses.MeanAbsoluteError(),
            "mape": tf.keras.losses.MeanAbsolutePercentageError(),
        }
        self.loss_fn = losses[loss_fn]
        self.num_episodes = num_episodes
        self.num_steps = num_steps if num_steps >= self.env.maxlen else self.env.maxlen
        self.gamma = gamma
        self.batch_size = batch_size
        loss: float = float("inf")
        no_improve_count: int = 0
        best_combined_metric: float = float("inf")

        for episode in range(self.num_episodes):
            print_progress_bar(episode + 1, self.num_episodes)
            self.env.reset()
            sum_rewards = 0
            epsilon = max(1 - episode / (self.num_episodes * 0.8), 0.01)

            for step in range(self.num_steps):
                state, action, reward, next_action, done = self.play_one_step(
                    x_data[step], y_data[step], epsilon
                )
                sum_rewards += reward if isinstance(reward, int) else reward[0]

                # Train if buffer has enough samples
                if len(self.replay_buffer) > self.batch_size:
                    loss = self._training_step()

                if done:
                    break

            combined_metric = loss - alpha * sum_rewards

            if combined_metric < best_combined_metric:
                best_combined_metric = combined_metric
                self.best_weights = self.model.get_weights()
                self.best_loss = loss
                no_improve_count = 0  # Reset counter on improvement
            else:
                no_improve_count += 1

            rewards.append(sum_rewards)

            # Logging
            if episode % (self.num_episodes // 10) == 0:
                print(
                    f"Episode: {episode}, Steps: {step+1}, Epsilon: {epsilon:.3f}, Loss: {loss:.2e}, Reward: {sum_rewards}, No Improve Count: {no_improve_count}"
                )

            # Early stopping condition
            if no_improve_count >= patience:
                print(
                    f"Early stopping at episode {episode} due to no improvement in {patience} episodes."
                )
                break

        # Save best model
        self.model.set_weights(self.best_weights)

    def __str__(self) -> str:
        """Return a human-readable string representation of the agent."""
        return (
            f"AutoQL (Env: {self.env.name}, Episodes: {self.num_episodes}, Steps: {self.num_steps})"
        )
```

</details>
